# Agentic RAG Chatbot - "Attention Is All You Need"

This notebook demonsrates an Agentic RAG (Retrieval-Augmented Generation) pipeline built with LangGraph. The system can autonomously decide wether to reactive information evaluate retrieved documents for relevance, rewrite queries when results are poor, and check its own answers for hallucinations.

**Data source:** [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) (Vaswani et al., 2017)

**Key patterns implemented:**
- Adaptive routing (retrieval vs. direct answer)
- Corrective RAG (document grading + query rewriting)
- Self-RAG (hallucination check + answer grading)

## 1. Setup and dependencies

Installing the required packages. Key components:
- **LangGraph** - state machine freamwork for the agentic pipeline
- **ChromaDB** - vector store for document embeddings
- **sentence-transformers** - multilingial embedding model
- **transforms + bitsandbytes** - local LLM inference with 4-bit quantization
- **PyMuPDF** - PDF text extraction

In [4]:
!pip install -q langgraph langchain langchain-community sentence-transformers chromadb pymupdf transformers bitsandbytes accelerate

## 2. Document loading

Downloading the original Transformer paper from arXiv and extracting raw text using PyMuPDF. This serves as our knowledge base - a single, well known paper is sufficient to demonstratte the full pipeline. The architecture is designed to scale to hundreds of documents without modifications.

In [6]:
!wget -q -O attention.pdf https://arxiv.org/pdf/1706.03762

import fitz
doc = fitz.open("attention.pdf")
print (f"Number of pages: {len(doc)}")
print(f"First 500 chatracters of the first page: {doc[0].get_text()[:500]}")

Number of pages: 15
First 500 chatracters of the first page: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K
